# Librerie

In [1]:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mpinaz/Thesis/blob/main/Model.ipynb)!pip install obspy
!git clone https://github.com/Mpinaz/Thesis.git

fatal: destination path 'Thesis' already exists and is not an empty directory.


In [2]:
from obspy import read, Stream, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.clients.fdsn.header import FDSNNoDataException

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit

import numpy as np
import pandas as pd

# KVG Catalog

## Stations

### Retrieve Coordinates

In [3]:
client = Client("https://geofon.gfz-potsdam.de")
tstart = UTCDateTime("2015-07-01 00:00:00")
tend   = UTCDateTime("2016-07-31 00:00:00")

inv = client.get_stations(network="X9",  starttime=tstart, endtime=tend, channel="*HN")
# print(inv)
nsta = np.shape(inv)[1]

# stla = np.zeros(nsta)
# stlo = np.zeros(nsta)
coordinates_for_station = {}
stname = []

for i in range(0,nsta):
    coordinates_for_station[inv[0][i].code] = [inv[0][i].latitude,inv[0][i].longitude]

## Catalog

In [4]:
catalog = pd.read_csv(
    '/content/Thesis/KVG_catalog.txt',
    sep=',',
    header=None,
    names=['time', 'lat', 'lon', 'depth_km', 'mag'],
)
catalog['time'] = pd.to_datetime(catalog['time'])
catalog['index'] = [f"Event {i}" for i in range(1,len(catalog)+1)]
catalog = catalog.set_index("index")


print(f"Loaded {len(catalog)} events")
print(f"Time range: {catalog['time'].min()} to {catalog['time'].max()}")
print(f"Magnitude range: {catalog['mag'].min():.1f} – {catalog['mag'].max():.1f}")
catalog.head()

Loaded 11209 events
Time range: 2015-08-03 14:56:47.810000+00:00 to 2016-07-04 21:46:56.760000+00:00
Magnitude range: 0.0 – 3.1


,time,lat,lon,depth_km,mag
index,,,,,
Event 1,2015-08-03 14:56:47.810000+00:00,56.051729,160.607292,30.49,1.42
Event 2,2015-08-05 02:33:26.280000+00:00,56.027136,160.655599,31.35,2.01
Event 3,2015-08-06 02:11:34.860000+00:00,56.038220,160.563005,29.36,1.97
Event 4,2015-08-06 08:01:51.920000+00:00,56.051261,160.605957,29.73,1.96
Event 5,2015-08-06 10:07:05.340000+00:00,56.067623,160.550732,43.55,1.86


## Phase Data

In [5]:
stations_for_event = {}
Event_per_station = {}
i=0
with open("/content/Thesis/KVG_phase_data.txt", "r", encoding="UTF-8") as f:
    date = -1
    line = f.readline()

    while line != "":
        check = line[:2]
        match check:
            case "X9":
                datas = line.split(",")
                #                                              NOME STAZIONE     P TIME   S TIME
                stations_for_event.setdefault(f"Event {i}", {})[datas[0][3:]] = (datas[1],datas[2])
                Event_per_station.setdefault(datas[0][3:], []).append(f"Event {i}")

            case "D0":
                pass

            case _:
                date = line.split(",")[0]
                i+=1
        line = f.readline()

# Model

## Pre e Post Events

In [28]:
# Event_per_station Dizionario dove per chiave ho il nome della stazione e valore la lista degli eventi registrati
Stazioni_da_usare = ["SV13", "SV6", "SV7", "IR2", "IR3", "IR4", "IR6"]

# Maschera per PRE-Eruzione
start = pd.Timestamp("2016-03-23", tz="UTC")
end   = pd.Timestamp("2016-04-20", tz="UTC")

mask = (catalog["time"] >= start) & (catalog["time"] <= end)
catalog_PRE = catalog[mask].index.tolist()
PRE_Eruption = []

for stazione in Stazioni_da_usare:
  PRE_Eruption += [(stazione, ev) for ev in catalog_PRE if ev in set(Event_per_station[stazione])]

# Maschera per POST-Eruzione
start = pd.Timestamp("2016-04-21", tz="UTC")
end   = pd.Timestamp("2016-06-05", tz="UTC")

mask = (catalog["time"] >= start) & (catalog["time"] <= end)
catalog_POST = catalog[mask].index.tolist()
POST_Eruption = []

for stazione in Stazioni_da_usare:
  POST_Eruption += [(stazione, ev) for ev in catalog_POST if ev in set(Event_per_station[stazione])]

print(len(PRE_Eruption),len(POST_Eruption))

1039 1036


## Retrieve Data and Filtering

In [29]:
# lunghezza temporale dell'onda
# 23 min per 2050
pre_time   = 2
Post_time  = 18

Events = []
Dataset_By_Event   = {}
Dataset_By_Station = {}
labels = []                       # 0 = pre, 1 = post
POST_set = set(E for (St, E) in POST_Eruption)
n = len(PRE_Eruption) + len(POST_Eruption)
for index, Tmp in enumerate(PRE_Eruption + POST_Eruption):
    St, E = Tmp
    T = UTCDateTime(stations_for_event[E][St][0])
    try:
        Reg = client.get_waveforms(
            network="X9",
            station=St,
            location="*",
            channel="*",
            starttime=T - pre_time,
            endtime=T + Post_time)

        pos = len(Events)                 # indice reale in Events
        Events.append((St, E, Reg))
        labels.append(1 if E in POST_set else 0)
        Dataset_By_Station.setdefault(St, []).append(pos)
        Dataset_By_Event.setdefault(E, []).append(pos)

    except FDSNNoDataException:
        print(f"Errore per {E} nella stazione {St}\n")
        continue
    if not (index + 1) % 50: print(f"{index + 1}: {((index + 1)/n * 100):.2f}%  Fatti. ")

50: 2.41%  Fatti. 
100: 4.82%  Fatti. 
150: 7.23%  Fatti. 
200: 9.64%  Fatti. 
250: 12.05%  Fatti. 
300: 14.46%  Fatti. 
350: 16.87%  Fatti. 
400: 19.28%  Fatti. 
450: 21.69%  Fatti. 
500: 24.10%  Fatti. 
550: 26.51%  Fatti. 
600: 28.92%  Fatti. 
650: 31.33%  Fatti. 
700: 33.73%  Fatti. 
750: 36.14%  Fatti. 
800: 38.55%  Fatti. 
850: 40.96%  Fatti. 
900: 43.37%  Fatti. 
950: 45.78%  Fatti. 
1000: 48.19%  Fatti. 
1050: 50.60%  Fatti. 
1100: 53.01%  Fatti. 
1150: 55.42%  Fatti. 
1200: 57.83%  Fatti. 
1250: 60.24%  Fatti. 
1300: 62.65%  Fatti. 
1350: 65.06%  Fatti. 
1400: 67.47%  Fatti. 
1450: 69.88%  Fatti. 
1500: 72.29%  Fatti. 
1550: 74.70%  Fatti. 
1600: 77.11%  Fatti. 
1650: 79.52%  Fatti. 
1700: 81.93%  Fatti. 
1750: 84.34%  Fatti. 
1800: 86.75%  Fatti. 
1850: 89.16%  Fatti. 
1900: 91.57%  Fatti. 
1950: 93.98%  Fatti. 
2000: 96.39%  Fatti. 
2050: 98.80%  Fatti. 


In [30]:
# Smoothing
for Station, E, st in Events:
  for tr in st:

    tr.detrend("demean")
    tr.detrend("linear")

    tr.filter(
        "bandpass",
        freqmin=1,
        freqmax=20,
        corners=4,
        zerophase=True)

## Dataset for the model

In [40]:
X = np.arange(len(Events))
y = np.array(labels)
groups_ev = np.array([E for (St, E, D) in Events])   # gruppo = evento

def split_per_gruppo(groups, seed=0):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr, tmp = next(gss1.split(X, y, groups))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    v, te = next(gss2.split(X[tmp], y[tmp], groups[tmp]))
    val, test = tmp[v], tmp[te]
    return tr, val, test

train_idx, val_idx, test_idx = split_per_gruppo(groups_ev)

# recupero dati
train = [Events[i] for i in train_idx]
val   = [Events[i] for i in val_idx]
test  = [Events[i] for i in test_idx]

In [41]:
len(train)

1518

In [32]:
class WaveDataset(Dataset):
    def __init__(self, events, labels, target_len=1000):
        self.events = events
        self.labels = labels
        self.target_len = target_len

    def __len__(self):
        return len(self.events)

    def __getitem__(self, i):
        St, E, D = self.events[i]
        arr = np.stack([tr.data[:self.target_len] for tr in D]).astype(np.float32)

        # normalizzazione per traccia
        arr = arr - arr.mean(axis=1, keepdims=True)
        std = arr.std(axis=1, keepdims=True)
        arr = arr / (std + 1e-8)

        x = torch.from_numpy(arr) # (3, 1000)
        y = torch.tensor(self.labels[i], dtype=torch.long)
        return x, y

In [11]:
class CNN(nn.Module):
    def __init__(self, in_channels=1, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),   # riduce l'asse temporale a 1
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):            # x: (batch, in_channels, lunghezza)
        x = self.features(x)         # (batch, 64, 1)
        x = x.flatten(1)             # (batch, 64)
        return self.classifier(x)

In [12]:
class RNN(nn.Module):
    def __init__(self, in_channels=1, hidden=64, n_layers=2, n_classes=2):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=in_channels,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(hidden * 2, n_classes)  # *2 per bidirezionale

    def forward(self, x):            # x: (batch, lunghezza, in_channels)
        out, (h, c) = self.rnn(x)
        x = out[:, -1, :]            # ultimo step temporale
        return self.classifier(x)